# AMR-Steward — GRPO Training Notebook

**RL environment for clinical antimicrobial stewardship.** Trains an LLM to prescribe the right antibiotic for drug-resistant bacterial infections — verified against EUCAST breakpoints and IDSA guidelines.

**Stack:** OpenEnv · TRL GRPOTrainer · Unsloth · HuggingFace Hub

---

### What this notebook does
1. Installs Unsloth + TRL GRPO
2. Trains `Qwen/Qwen3-1.5B` across 3 curriculum stages using multi-head GRPO
3. Plots reward curves across all stages
4. Pushes the trained LoRA adapter to HuggingFace Hub

**Reward signal is fully verifiable (no LLM judges):**
- R0: allergy safety gate
- R1: microbiological activity (EUCAST breakpoints)
- R2: guideline concordance (IDSA first-line)
- R3: stewardship (narrowest effective drug)
- R4: dose correctness (renal-adjusted)
- R5: tool efficiency
- quality_ratio = agent_score / optimal_score (RLVR oracle)

**Runtime:** T4 GPU (~90 min) or A100 (~30 min)


## 1. Install Dependencies

In [ ]:
# Install Unsloth (fast GRPO training with 2-5x speedup)
# Using the colab-compatible build
import subprocess, sys

# Detect CUDA version for correct Unsloth wheel
result = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], 
                       capture_output=True, text=True)
print("GPU:", result.stdout.strip() if result.returncode == 0 else "not detected")

In [ ]:
%%capture
# Unsloth + TRL GRPO stack
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || \
 pip install unsloth
!pip install "trl>=0.19" "accelerate>=0.34" "peft>=0.11" "datasets>=2.19" \
             "transformers>=4.51" "huggingface_hub>=0.23"
# OpenEnv framework
!pip install "openenv-core>=0.2.3"

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Clone Repo

In [ ]:
!git clone https://github.com/saaheerpurav/amr-steward.git
%cd /content/amr-steward
!ls

In [ ]:
# Verify environment works before training
import sys
sys.path.insert(0, '/content/amr-steward')

from env import AMREnvironment, AMRAction

env = AMREnvironment()
obs = env.reset(curriculum_level=1)
print("Environment loaded successfully.")
print(obs.patient_text[:300])

## 3. HuggingFace Authentication

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Store your HF token in Colab Secrets (Key icon → Add secret HF_TOKEN)
try:
    token = userdata.get('HF_TOKEN')
    login(token=token, add_to_git_credential=False)
    print("Logged in via Colab secret.")
except Exception:
    # Fallback: paste token manually
    login()  # prompts interactively

## 4. Training Configuration

Change `MODEL_NAME` and `SAMPLES_*` to adjust training scale.

| Hardware | Recommended model | Samples/stage | Time |
|----------|-------------------|---------------|------|
| T4 (16 GB) | `Qwen/Qwen3-1.5B` | 64 / 32 / 16 | ~60 min |
| A100 (80 GB) | `Qwen/Qwen3-4B` | 256 / 128 / 64 | ~90 min |

In [ ]:
# ── Training hyperparameters ──────────────────────────────────────────────────
MODEL_NAME       = "Qwen/Qwen3-1.5B"   # change to Qwen/Qwen3-4B for A100
HF_REPO_ID       = "saaheerpurav/amr-steward-model"
OUTPUT_DIR       = "/content/checkpoints/amr-grpo"

SAMPLES_STAGE1   = 64    # susceptible organisms only (level 1)
SAMPLES_STAGE2   = 32    # + ESBL / MRSA / VRE (level 2)
SAMPLES_STAGE3   = 16    # + MDR / renal failure / allergies (level 3)

EPOCHS           = 1
BATCH_SIZE       = 1
GRAD_ACCUM       = 8
NUM_GENERATIONS  = 4     # GRPO group size
LEARNING_RATE    = 5e-6
MAX_COMPLETION   = 512   # tokens — enough for 2-3 INVESTIGATE + COMMIT
TEMPERATURE      = 0.7
LORA_R           = 16
LORA_ALPHA       = 32
MAX_SEQ_LEN      = 2048
LOAD_IN_4BIT     = True  # set False for A100 bfloat16 full precision

print(f"Model: {MODEL_NAME}")
print(f"Total samples: {SAMPLES_STAGE1 + SAMPLES_STAGE2 + SAMPLES_STAGE3}")

## 5. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,           # auto-detect bfloat16/float16
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model.print_trainable_parameters()

## 6. Reward Functions

Three independent GRPO reward heads — no LLM judge anywhere:

| Head | Signal | Timescale |
|------|--------|-----------|
| `format_reward_fn` | R6: clean COMMIT line | fast (per token) |
| `process_reward_fn` | R5: diverse tool use with budget | dense (per step) |
| `terminal_reward_fn` | quality_ratio oracle = agent_score / optimal_score | sparse (terminal) |

In [ ]:
import json, sys
from dataclasses import asdict
from pathlib import Path

sys.path.insert(0, '/content/amr-steward')

from env import AMREnvironment, AMRAction
from env.models import PatientCase
from env.reward import (
    parse_prescription_from_text,
    parse_tool_calls_from_text,
    R5_tool_efficiency,
    R6_format,
)


SYSTEM_PROMPT = """You are an antimicrobial stewardship AI. Prescribe the narrowest effective antibiotic.

INVESTIGATE tools (optional, costs 1 budget each):
  INVESTIGATE: {"tool": "interpret_resistance", "arg": "<drug>"}
  INVESTIGATE: {"tool": "check_guideline", "arg": "<syndrome>"}
  INVESTIGATE: {"tool": "assess_patient_factors"}

When ready, output EXACTLY this one line and stop:
  COMMIT: {"drug": "<name>", "dose": "<dose>", "duration": "<days>", "justification": "<one sentence>"}

Do not add any text before or after the COMMIT line."""

_BUDGET_BY_LEVEL = {1: 5, 2: 4, 3: 3}


def _completion_to_text(c):
    if isinstance(c, str): return c
    if isinstance(c, list):
        parts = []
        for item in c:
            if isinstance(item, dict): parts.append(item.get("content", ""))
            elif isinstance(item, str): parts.append(item)
        return "\n".join(parts)
    return str(c)


def _parse_investigate_action(raw_line):
    try:
        parsed = json.loads(raw_line.strip())
        tool_name = parsed.get("tool")
        return (tool_name, parsed.get("arg") or None) if isinstance(tool_name, str) and tool_name else None
    except Exception:
        return None


def _extract_tool_type(tc):
    try: return json.loads(tc.strip()).get("tool", tc)
    except Exception: return tc.split()[0] if tc.split() else "unknown"


def _score_with_env(completion_text, patient_payload, level):
    patient = PatientCase(**json.loads(patient_payload))
    env = AMREnvironment()
    env.reset(curriculum_level=int(level), patient=patient)
    cumulative = 0.0
    for raw_line in parse_tool_calls_from_text(completion_text):
        if env._state.done: break
        parsed = _parse_investigate_action(raw_line)
        if parsed is None: continue
        tool_name, tool_arg = parsed
        try:
            obs = env.step(AMRAction(action_type="INVESTIGATE", tool_name=tool_name, tool_arg=tool_arg))
            if obs.reward: cumulative += obs.reward
        except Exception:
            continue
    if not env._state.done:
        prescription = parse_prescription_from_text(completion_text)
        if prescription:
            try:
                obs = env.step(AMRAction(action_type="COMMIT", prescription=prescription))
                if obs.reward: cumulative += obs.reward
            except Exception:
                pass
    return cumulative


def format_reward_fn(prompts, completions, **kwargs):
    """Head 1: penalise verbose completions (fast feedback)."""
    return [float(R6_format(_completion_to_text(c))) * 0.05 for c in completions]


def process_reward_fn(prompts, completions, patient_json, curriculum_level=None, **kwargs):
    """Head 2: diverse tool use relative to budget (dense)."""
    rewards = []
    levels = list(curriculum_level) if curriculum_level is not None else [1] * len(completions)
    for completion, level in zip(completions, levels):
        text = _completion_to_text(completion)
        tool_calls = parse_tool_calls_from_text(text)
        budget_total = _BUDGET_BY_LEVEL.get(int(level), 5)
        unique_types = len({_extract_tool_type(tc) for tc in tool_calls})
        budget_spent = len(tool_calls)
        budget_remaining = max(0, budget_total - budget_spent)
        rewards.append(float(R5_tool_efficiency(unique_types, budget_spent, budget_remaining, budget_total)))
    return rewards


def terminal_reward_fn(prompts, completions, patient_json, curriculum_level=None, **kwargs):
    """Head 3: cumulative env reward — dense shaping + RLVR quality_ratio (sparse)."""
    levels = list(curriculum_level) if curriculum_level is not None else [1] * len(completions)
    rewards = []
    for completion, payload, level in zip(completions, patient_json, levels):
        try:
            rewards.append(float(_score_with_env(_completion_to_text(completion), payload, int(level))))
        except Exception:
            rewards.append(0.0)
    return rewards


print("Reward functions defined.")

## 7. Dataset Builder

In [ ]:
from datasets import Dataset
from typing import Any


def _render_prompt(obs) -> str:
    user_content = obs.patient_text
    if obs.tool_results:
        user_content += "\n\nINVESTIGATION RESULTS:\n" + "\n---\n".join(obs.tool_results)
    if obs.world_model_rankings:
        user_content += f"\n\n{obs.world_model_rankings}"
    user_content += f"\n\nINVESTIGATION BUDGET REMAINING: {obs.budget_remaining}"
    if obs.budget_remaining == 0:
        user_content += "\n\nYOU MUST COMMIT NOW."
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    # Qwen3 chat template with thinking disabled — triggers instruction-following mode
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )


def build_dataset(level: int, num_samples: int) -> Dataset:
    env = AMREnvironment()
    rows: list[dict[str, Any]] = []
    for idx in range(num_samples):
        obs = env.reset(curriculum_level=level)
        patient = env.current_patient
        rows.append({
            "prompt": _render_prompt(obs),
            "patient_json": json.dumps(asdict(patient)),
            "curriculum_level": level,
            "case_id": f"level{level}-case{idx}",
        })
    return Dataset.from_list(rows)


print("Dataset builder ready.")

## 8. Training Loop

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import os

all_log_history = {}  # stage_label -> list of log dicts


def train_stage(level: int, num_samples: int, stage_label: str):
    print(f"\n{'='*60}")
    print(f"  {stage_label}: curriculum_level={level}, samples={num_samples}")
    print(f"{'='*60}")

    dataset = build_dataset(level, num_samples)
    print(f"  Dataset built: {len(dataset)} cases")

    stage_output = f"{OUTPUT_DIR}/{stage_label}"
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    config = GRPOConfig(
        output_dir=stage_output,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        bf16=bf16,
        fp16=torch.cuda.is_available() and not bf16,
        logging_steps=1,
        save_steps=50,
        save_total_limit=1,
        report_to="none",
        max_completion_length=MAX_COMPLETION,
        num_generations=NUM_GENERATIONS,
        temperature=TEMPERATURE,
        log_completions=True,
        use_vllm=False,
    )

    trainer = GRPOTrainer(
        model=model,
        reward_funcs=[format_reward_fn, process_reward_fn, terminal_reward_fn],
        args=config,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
    trainer.train()

    history = trainer.state.log_history
    all_log_history[stage_label] = history

    Path(stage_output).mkdir(parents=True, exist_ok=True)
    (Path(stage_output) / "log_history.json").write_text(
        json.dumps(history, indent=2), encoding="utf-8"
    )
    return trainer


print("Training loop defined.")

## 9. Stage 1 — Susceptible Organisms

Simple cases: susceptible K. pneumoniae, E. coli, S. aureus, Enterococcus. Normal renal function. No allergies. Budget = 5 tools.

In [ ]:
trainer_s1 = train_stage(level=1, num_samples=SAMPLES_STAGE1, stage_label="stage1")

In [ ]:
# Quick reward summary for Stage 1
history = all_log_history["stage1"]
train_steps = [h for h in history if "reward" in h and "epoch" in h and "train_runtime" not in h]
if train_steps:
    rewards = [h["reward"] for h in train_steps]
    print(f"Stage 1 | Steps: {len(train_steps)} | "
          f"Initial: {rewards[0]:.3f} | Peak: {max(rewards):.3f} | Final: {rewards[-1]:.3f}")

## 10. Stage 2 — Resistant / MDR Organisms

Harder cases: ESBL-producing E. coli, MRSA, VRE. Mild–moderate renal impairment. Budget = 4 tools.

In [ ]:
trainer_s2 = train_stage(level=2, num_samples=SAMPLES_STAGE2, stage_label="stage2")

In [ ]:
history = all_log_history["stage2"]
train_steps = [h for h in history if "reward" in h and "epoch" in h and "train_runtime" not in h]
if train_steps:
    rewards = [h["reward"] for h in train_steps]
    print(f"Stage 2 | Steps: {len(train_steps)} | "
          f"Initial: {rewards[0]:.3f} | Peak: {max(rewards):.3f} | Final: {rewards[-1]:.3f}")

## 11. Stage 3 — MDR + Severe Renal Failure + Allergies

Hardest cases: CRE, XDR Pseudomonas, VISA. Severe renal impairment (CrCl <30). Penicillin / cephalosporin allergies. Budget = 3 tools.

In [ ]:
trainer_s3 = train_stage(level=3, num_samples=SAMPLES_STAGE3, stage_label="stage3")

In [ ]:
history = all_log_history["stage3"]
train_steps = [h for h in history if "reward" in h and "epoch" in h and "train_runtime" not in h]
if train_steps:
    rewards = [h["reward"] for h in train_steps]
    print(f"Stage 3 | Steps: {len(train_steps)} | "
          f"Initial: {rewards[0]:.3f} | Peak: {max(rewards):.3f} | Final: {rewards[-1]:.3f}")

## 12. Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def extract_reward_series(label):
    history = all_log_history.get(label, [])
    steps, rewards = [], []
    for h in history:
        if "reward" in h and "epoch" in h and "train_runtime" not in h:
            steps.append(h["step"])
            rewards.append(h["reward"])
    return steps, rewards


fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle("AMR-Steward: GRPO Reward Across Curriculum Stages", fontsize=14, fontweight='bold')

stage_configs = [
    ("stage1", "Stage 1 — Susceptible", "#2196F3"),
    ("stage2", "Stage 2 — Resistant/MDR", "#FF9800"),
    ("stage3", "Stage 3 — MDR + Renal + Allergies", "#F44336"),
]

for ax, (label, title, color) in zip(axes, stage_configs):
    steps, rewards = extract_reward_series(label)
    if steps:
        ax.plot(steps, rewards, marker='o', color=color, linewidth=2, markersize=6)
        ax.axhline(y=max(rewards), color=color, linestyle='--', alpha=0.4, label=f"Peak {max(rewards):.3f}")
        ax.fill_between(steps, rewards, alpha=0.1, color=color)
        # Annotations
        ax.annotate(f"{rewards[0]:.3f}", (steps[0], rewards[0]), textcoords="offset points",
                    xytext=(5, 5), fontsize=9, color=color)
        ax.annotate(f"{rewards[-1]:.3f}", (steps[-1], rewards[-1]), textcoords="offset points",
                    xytext=(-20, 5), fontsize=9, color=color)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Mean Reward" if ax == axes[0] else "")
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: reward_curves.png")

## 13. Save Model and Push to HuggingFace Hub

In [ ]:
from pathlib import Path

FINAL_OUTPUT = f"{OUTPUT_DIR}/final"
Path(FINAL_OUTPUT).mkdir(parents=True, exist_ok=True)

# Save LoRA adapter
trainer_s3.save_model(FINAL_OUTPUT)
tokenizer.save_pretrained(FINAL_OUTPUT)

print(f"Model saved to {FINAL_OUTPUT}")
print("Files:", [f.name for f in Path(FINAL_OUTPUT).iterdir()])

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HF_REPO_ID, repo_type="model", exist_ok=True)

# Push LoRA adapter weights
api.upload_folder(
    folder_path=FINAL_OUTPUT,
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="GRPO training: 3-head reward (quality_ratio oracle + JEPA + dense shaping)",
)

# Also push reward curves
api.upload_file(
    path_or_fileobj="reward_curves.png",
    path_in_repo="reward_curves.png",
    repo_id=HF_REPO_ID,
    repo_type="model",
)

print(f"Model pushed to https://huggingface.co/{HF_REPO_ID}")

## 14. Inference Demo

Compare untrained vs trained model on the same patient case.

In [ ]:
from unsloth import FastLanguageModel as FLM

# Enable inference mode (2x faster generation)
FLM.for_inference(model)


def run_episode(patient_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": patient_text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.3,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Use the live HF Space environment
import requests

BASE = "https://saaheerpurav-amr-steward.hf.space"

try:
    resp = requests.post(f"{BASE}/reset", json={"curriculum_level": 1}, timeout=10)
    resp.raise_for_status()
    patient_text = resp.json()["observation"]["patient_text"]
    print("Patient case from live environment:")
    print(patient_text)
except Exception as e:
    print(f"Environment call failed ({e}), using local case:")
    env = AMREnvironment()
    obs = env.reset(curriculum_level=1)
    patient_text = obs.patient_text
    print(patient_text)

print("\nTrained model response:")
print(run_episode(patient_text))

## Summary

| Component | Detail |
|-----------|--------|
| **Model** | Qwen3-1.5B + LoRA (r=16) via Unsloth (T4) · Qwen3-4B on A10G for full run |
| **Training** | Multi-head GRPO: format (R6) + process (R5) + terminal (quality_ratio) |
| **Reward oracle** | quality_ratio = agent_score / compute_optimal_prescription() — no LLM judge |
| **World model** | JEPA pre-trained on 5,201 synthetic episodes — guides investigation strategy |
| **Curriculum** | 3 stages: susceptible → MDR → MDR + renal + allergies |
| **Live environment** | https://divyanshb06-amrsteward.hf.space |
| **Trained model** | https://huggingface.co/saaheerpurav/amr-steward-model |

**Results (Qwen3-4B, A10G):**

| Stage | Peak Reward | vs Random baseline (~0.05–0.10) |
|-------|-------------|----------------------------------|
| Stage 1 — Susceptible | **0.840** | +0.74 |
| Stage 2 — Resistant/MDR | **0.790** | +0.69 |
| Stage 3 — MDR + Renal + Allergies | **0.707** | +0.61 |

The model learns to investigate resistance data, consult IDSA guidelines, and adjust for renal function — all without an LLM judge.